# Cartes d'identité des fichiers CSV et SQL

**Objectif** : avant d'exploiter des données, on vérifie *ce qu'on a reçu*. Pour chaque fichier de `datas/`,
ce notebook produit une **carte d'identité** en markdown : origine, taille, horaires de traitement, encodage,
nombre de lignes et de colonnes, type de chaque colonne, et min / max / moyenne / médiane des colonnes
numériques et dates.

| | |
|---|---|
| **Entrée** | les fichiers bruts de `datas/` : `*.csv` et scripts SQL `*.sql*` (couche **Bronze** : lus, jamais modifiés) |
| **Sortie** | une carte par fichier : `artifacts/ingestion/identities/AAAAMMJJ/<fichier>_HHMMSS.md` |

## Principe de généralisation

Chaque format a **sa propre fonction de lecture**, mais toutes ramènent la source à la **même forme commune** :
un `DataFrame` pandas + un dictionnaire `{colonne: type}`. Tout le reste (statistiques, mise en forme, écriture)
est **commun** et ne sait pas d'où viennent les données.

```
LECTURE (propre au format)                  FORME COMMUNE                 COMMUN À TOUS LES FORMATS
fichier.csv ─► build_identity_card     ─┐
               (read_csv + infer_type)  ├─► (DataFrame, {col: type}) ─► render_header + render_table ─► write_card ─► carte .md
fichier.sql ─► build_sql_identity_card ─┘   (une par table)
               (sqlglot + sql_column_type)
```

Ajouter un format (JSON, Excel…) = écrire **une** fonction de lecture qui renvoie cette forme commune.

## Plan
1. Imports et paramètres
2. Fonctions communes à tous les formats
3. Lecture des CSV
4. Lecture des scripts SQL
5. Écriture de la carte
6. Auto-contrôle des fonctions
7. Génération des cartes

## 1. Imports et paramètres

- `csv` : détection du séparateur ; `datetime` : horodatage ; `pathlib.Path` : chemins de fichiers.
- `pandas` : lecture des CSV et calcul des statistiques.
- `sqlglot` : **parseur SQL**, transforme le texte du script en objets Python sans l'exécuter.
- `DATA_DIR` : dossier d'entrée ; `OUT_DIR` : dossier de sortie des cartes. Ce sont les deux seuls chemins du notebook.

In [ ]:
import csv
from datetime import datetime
from pathlib import Path

import pandas as pd
import sqlglot
from sqlglot import exp

DATA_DIR = Path("datas")
OUT_DIR = Path("artifacts/ingestion/identities")

## 2. Fonctions communes à tous les formats

Ces fonctions servent pour les CSV **et** pour le SQL.

### `detect_charset(path)` : encodage du fichier

| | |
|---|---|
| **Entrée** | `path` : chemin du fichier |
| **Sortie** | `str` : `"utf-8-sig"` (UTF-8 avec BOM), `"utf-8"`, `"cp1252"` ou `"latin-1"` |

**Comment** : lit les octets du fichier et essaie de les décoder dans l'ordre UTF-8 → Windows-1252.
Le premier qui réussit est le bon. Latin-1 sert de dernier recours car il décode n'importe quel octet.
Un mauvais encodage abîmerait les accents (« dÃ©faut » au lieu de « défaut »).

In [ ]:
def detect_charset(path):
    """Encodage du fichier : UTF-8 (avec ou sans BOM), sinon Windows-1252, sinon Latin-1."""
    raw = Path(path).read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    for enc in ("utf-8", "cp1252"):
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            pass
    return "latin-1"  # décode toujours

### `human_size(n_bytes)` : taille lisible

| | |
|---|---|
| **Entrée** | `n_bytes` : taille en octets (`int`) |
| **Sortie** | `str` au format `9999.99 Unité` : Octets, Ko, Mo ou Go |

**Comment** : divise par 1024 tant que la valeur dépasse 1024. Exemple : `human_size(8323471)` → `"7.94 Mo"`.

In [ ]:
def human_size(n_bytes):
    """9999.99 Octets | Ko | Mo | Go (base 1024)."""
    size = float(n_bytes)
    for unit in ("Octets", "Ko", "Mo"):
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} Go"

### `column_stats(series, col_type)` : statistiques d'une colonne

| | |
|---|---|
| **Entrée** | `series` : la colonne (pandas `Series`) ; `col_type` : son type (`INTEGER`, `FLOAT`, `DATE`, `DATETIME`, `TIME`) |
| **Sortie** | `dict` : `{"min", "max", "mean", "median"}` (mean et median vides pour les dates) |

**Comment** :
- numérique : min, max, moyenne (arrondie à 3 décimales), médiane ;
- date / datetime : conversion en vraies dates (`pd.to_datetime`) puis min et max ;
- heure (`HH:MM`) : min et max directement, car l'ordre alphabétique est l'ordre chronologique.

In [ ]:
def column_stats(series, col_type):
    """Min / Max (+ Mean / Median pour les numériques)."""
    if col_type in ("INTEGER", "FLOAT"):
        return {"min": series.min(), "max": series.max(), "mean": round(series.mean(), 3), "median": series.median()}
    # TIME au format HH:MM(:SS) : l'ordre alphabétique est l'ordre chronologique
    values = series.dropna() if col_type == "TIME" else pd.to_datetime(series)
    if col_type == "DATE":
        values = values.dt.date
    return {"min": values.min(), "max": values.max(), "mean": "", "median": ""}

### `render_header(path, begin, end, charset)` : en-tête de la carte

| | |
|---|---|
| **Entrée** | `path` : chemin du fichier ; `begin` / `end` : heures de début et de fin (`datetime`) ; `charset` : encodage |
| **Sortie** | `list[str]` : lignes markdown (titre, Source, Filesize, ProcessBeginAt, ProcessEndAt, Charset) |

Identique pour tous les formats.

In [ ]:
def render_header(path, begin, end, charset):
    """En-tête commun : source, taille, horaires, encodage."""
    return [
        f"# Carte d'identité : {path.name}",
        "",
        f"- Source: {path}",
        f"- Filesize: {human_size(path.stat().st_size)}",
        f"- ProcessBeginAt: {begin:%H:%M:%S}",
        f"- ProcessEndAt: {end:%H:%M:%S}",
        f"- Charset: {charset.upper()}",
    ]

### `render_table(df, types, level="##")` : corps de la carte pour une table

| | |
|---|---|
| **Entrée** | `df` : les données (`DataFrame`) ; `types` : `{colonne: type}` ; `level` : niveau de titre markdown (`##` pour un CSV, `###` dans une section de table SQL) |
| **Sortie** | `list[str]` : lignes markdown (Rows / Cols Number, tableau des colonnes, tableau des statistiques) |

**C'est le cœur de la généralisation** : la fonction ne sait pas si `df` vient d'un CSV ou d'un SQL.
Les statistiques sont calculées seulement pour les types de `STAT_TYPES` (numériques et dates) et seulement
si la colonne contient des valeurs.

In [ ]:
STAT_TYPES = ("INTEGER", "FLOAT", "DATE", "DATETIME", "TIME")


def render_table(df, types, level="##"):
    """Lignes/colonnes, liste des colonnes typées et stats des colonnes numériques et dates."""
    # une colonne déclarée sans aucune valeur (ex. DEFAULT NOW() en SQL) n'a pas de stats
    stats = {col: column_stats(df[col], t) for col, t in types.items()
             if t in STAT_TYPES and col in df.columns and df[col].notna().any()}
    return [
        f"- Rows Number: {len(df)}",
        f"- Cols Number: {len(types)}",
        "",
        f"{level} Column list",
        "",
        "| Colname | Infered Type |",
        "| ------- | ------------ |",
        *[f"| {col} | {t} |" for col, t in types.items()],
        "",
        f"{level} Numeric and date columns",
        "",
        "| Colname | Min value | Max value | Mean value | Median value |",
        "| ------- | --------- | --------- | ---------- | ------------ |",
        *[f"| {col} | {s['min']} | {s['max']} | {s['mean']} | {s['median']} |" for col, s in stats.items()],
        "",
    ]

## 3. Lecture des CSV

Pour un CSV, ni le séparateur ni les types ne sont donnés : on les **détecte**.

### `detect_separator(path, charset, n_lines=20)` : séparateur du CSV

| | |
|---|---|
| **Entrée** | `path` : chemin du CSV ; `charset` : encodage (sortie de `detect_charset`) ; `n_lines` : nombre de lignes analysées |
| **Sortie** | `str` : le séparateur, parmi `,` `;` tabulation `\|` |

**Comment** : `csv.Sniffer` (bibliothèque standard) analyse les premières lignes et repère le caractère qui découpe
chaque ligne en un même nombre de champs. Le séparateur n'est pas toujours une virgule.

In [ ]:
def detect_separator(path, charset, n_lines=20):
    """Séparateur deviné sur les premières lignes (, ; tabulation |)."""
    with open(path, encoding=charset, newline="") as f:
        sample = "".join(f.readline() for _ in range(n_lines))
    return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter

### `infer_type(name, series)` : type d'une colonne CSV

| | |
|---|---|
| **Entrée** | `name` : nom de la colonne ; `series` : ses valeurs |
| **Sortie** | `str` : `IDENTITY`, `BOOLEAN`, `INTEGER`, `FLOAT`, `DATE`, `DATETIME`, `TIME` ou `TEXT` |

**Comment** : règles appliquées dans l'ordre, la première qui correspond gagne.
1. nom `id` ou terminé par `_id` → `IDENTITY` (un identifiant, pas une mesure) ;
2. nombres uniquement 0/1 → `BOOLEAN` ;
3. nombres entiers → `INTEGER`, sinon `FLOAT` ;
4. texte dont **toutes** les valeurs respectent un format de `DATE_FORMATS` → `DATE`, `DATETIME` ou `TIME` ;
5. sinon → `TEXT`.

Les valeurs vides sont ignorées. Pour reconnaître un autre format de date (ex. `01/06/2025`), il suffit de l'ajouter à `DATE_FORMATS`.

In [ ]:
DATE_FORMATS = [("DATE", "%Y-%m-%d"), ("DATETIME", "%Y-%m-%d %H:%M:%S"), ("TIME", "%H:%M"), ("TIME", "%H:%M:%S")]


def infer_type(name, series):
    """IDENTITY, BOOLEAN, INTEGER, FLOAT, DATE, DATETIME, TIME ou TEXT."""
    if name == "id" or name.endswith("_id"):
        return "IDENTITY"
    values = series.dropna()
    if pd.api.types.is_numeric_dtype(series):
        if len(values) and values.isin([0, 1]).all():
            return "BOOLEAN"
        return "INTEGER" if pd.api.types.is_integer_dtype(series) or (values % 1 == 0).all() else "FLOAT"
    for type_name, fmt in DATE_FORMATS:
        if len(values) and pd.to_datetime(values, format=fmt, errors="coerce").notna().all():
            return type_name
    return "TEXT"

### `build_identity_card(path)` : carte d'un CSV

| | |
|---|---|
| **Entrée** | `path` : chemin du CSV |
| **Sortie** | `tuple` : (carte markdown `str`, heure de début `datetime`) |

**Étapes** : heure de début → `detect_charset` → `detect_separator` → `pd.read_csv` → `infer_type` sur chaque colonne
→ **forme commune** `(df, types)` → `render_table` → heure de fin → `render_header` + corps.

In [ ]:
def build_identity_card(path):
    """Lit le CSV et renvoie (carte markdown, heure de début)."""
    path = Path(path).resolve()
    begin = datetime.now()
    charset = detect_charset(path)
    sep = detect_separator(path, charset)
    df = pd.read_csv(path, sep=sep, encoding=charset)
    types = {col: infer_type(col, df[col]) for col in df.columns}
    body = render_table(df, types)
    end = datetime.now()
    return "\n".join(render_header(path, begin, end, charset) + body), begin

## 4. Lecture des scripts SQL

Un script SQL n'est pas un tableau : c'est une suite d'instructions. `sqlglot` le **parse** (le découpe en objets
Python) sans l'exécuter. Ici les types ne sont pas devinés : ils sont **déclarés** dans les `CREATE TABLE`.

### `sql_column_type(col)` : type d'une colonne SQL

| | |
|---|---|
| **Entrée** | `col` : une définition de colonne d'un `CREATE TABLE` (objet `sqlglot.exp.ColumnDef`) |
| **Sortie** | `str` : même vocabulaire que `infer_type` (`IDENTITY`, `INTEGER`, `FLOAT`…) |

**Comment** :
- clé primaire (`PRIMARY KEY`), clé étrangère (`REFERENCES`) ou nom en `_id` → `IDENTITY` (même règle que pour les CSV) ;
- sinon le type déclaré est traduit avec `SQL_TYPES` (`INT` → `INTEGER`, `NUMERIC` → `FLOAT`, `TIMESTAMPTZ` → `DATETIME`…) ;
- tout type absent du dictionnaire (`VARCHAR`, `TEXT`…) → `TEXT`.

In [ ]:
# Types SQL déclarés -> nos types (le reste : VARCHAR, TEXT... -> TEXT)
SQL_TYPES = {
    "INT": "INTEGER", "SMALLINT": "INTEGER", "BIGINT": "INTEGER",
    "DECIMAL": "FLOAT", "FLOAT": "FLOAT", "DOUBLE": "FLOAT",
    "DATE": "DATE", "TIMESTAMP": "DATETIME", "TIMESTAMPTZ": "DATETIME", "TIME": "TIME",
    "BOOLEAN": "BOOLEAN",
}


def sql_column_type(col):
    """Type d'une colonne de CREATE TABLE : IDENTITY pour clé primaire/étrangère ou *_id, sinon type déclaré."""
    is_key = any(isinstance(c.kind, (exp.PrimaryKeyColumnConstraint, exp.Reference)) for c in col.constraints)
    if is_key or col.name == "id" or col.name.endswith("_id"):
        return "IDENTITY"
    return SQL_TYPES.get(col.kind.this.name, "TEXT")

### `parse_sql_tables(sql_text)` : tables d'un script SQL

| | |
|---|---|
| **Entrée** | `sql_text` : le texte du script (`str`) |
| **Sortie** | `dict` : `{nom_table: (DataFrame, {colonne: type})}`, soit la **forme commune**, une fois par table |

**Comment** : `sqlglot.parse(..., read="postgres")` renvoie la liste des instructions. On en garde deux sortes :
- `CREATE TABLE` → noms et types des colonnes (via `sql_column_type`) ;
- `INSERT` → chaque ligne de `VALUES` devient une ligne du DataFrame (`to_py()` convertit `'MACH-01'` en texte, `770` en entier, `NULL` en `None`).

Le reste (`CREATE INDEX`, `BEGIN`, `COMMIT`…) est ignoré. Le nombre de tables n'est pas fixé, et plusieurs `INSERT` vers une même table sont regroupés.
Les colonnes déclarées mais jamais remplies par un `INSERT` (ex. `created_at DEFAULT NOW()`) apparaissent dans la carte sans statistiques.

In [ ]:
def parse_sql_tables(sql_text):
    """{table: (DataFrame des INSERT, {colonne: type})} à partir d'un script PostgreSQL (lu, jamais exécuté)."""
    types, rows = {}, {}
    for stmt in sqlglot.parse(sql_text, read="postgres"):
        if isinstance(stmt, exp.Create) and stmt.kind == "TABLE":
            name = stmt.this.this.name
            types[name] = {c.name: sql_column_type(c) for c in stmt.this.expressions if isinstance(c, exp.ColumnDef)}
            rows[name] = []
        elif isinstance(stmt, exp.Insert):
            name = stmt.this.this.name
            cols = [c.name for c in stmt.this.expressions]
            rows[name] += [dict(zip(cols, (v.to_py() for v in row.expressions))) for row in stmt.expression.expressions]
    return {name: (pd.DataFrame(rows[name]), types[name]) for name in types}

### `build_sql_identity_card(path)` : carte d'un script SQL

| | |
|---|---|
| **Entrée** | `path` : chemin du script SQL |
| **Sortie** | `tuple` : (carte markdown `str`, heure de début `datetime`) |

**Étapes** : heure de début → `detect_charset` → lecture du texte → `parse_sql_tables` → pour **chaque table**,
une section `## Table : <nom>` produite par `render_table` (la même que pour les CSV) → heure de fin → `render_header` + sections.

In [ ]:
def build_sql_identity_card(path):
    """Lit le script SQL et renvoie (carte markdown avec une section par table, heure de début)."""
    path = Path(path).resolve()
    begin = datetime.now()
    charset = detect_charset(path)
    body = []
    for name, (df, types) in parse_sql_tables(path.read_text(encoding=charset)).items():
        body += ["", f"## Table : {name}", ""] + render_table(df, types, level="###")
    end = datetime.now()
    return "\n".join(render_header(path, begin, end, charset) + body), begin

## 5. Écriture de la carte

### `write_card(path)` : choisir le bon lecteur et écrire le fichier

| | |
|---|---|
| **Entrée** | `path` : chemin d'un fichier de `datas/` |
| **Sortie** | `Path` : chemin de la carte écrite |

**Comment** :
- choisit la fonction de lecture selon l'extension : `.sql` → `build_sql_identity_card`, sinon → `build_identity_card` ;
- construit le chemin `artifacts/ingestion/identities/AAAAMMJJ/<nom>_HHMMSS.md` à partir de l'heure de début et du nom du fichier source (`machine.sql.txt` → `machine`) ;
- crée le dossier du jour s'il n'existe pas et écrit la carte. **N'écrit jamais dans `datas/`**.

In [ ]:
def write_card(path):
    """Écrit la carte dans artifacts/ingestion/identities/AAAAMMJJ/<nom>_HHMMSS.md."""
    path = Path(path)
    build = build_sql_identity_card if ".sql" in path.suffixes else build_identity_card
    card, begin = build(path)
    out = OUT_DIR / f"{begin:%Y%m%d}" / f"{path.name.split('.')[0]}_{begin:%H%M%S}.md"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(card, encoding="utf-8")
    return out


## 6. Auto-contrôle des fonctions

Tests sur de **petits exemples dont on connaît la réponse**. Si une fonction est cassée, un `assert` lève une erreur ;
sinon la cellule affiche `OK`.
- `human_size` : conversions d'unités ;
- `infer_type` : un exemple par type ;
- `parse_sql_tables` : un mini-script SQL (types déclarés, nombre de lignes, colonne sans valeur, `NULL`, moyenne).

In [ ]:
assert human_size(500) == "500.00 Octets"
assert human_size(2048) == "2.00 Ko"
assert human_size(8323471) == "7.94 Mo"
assert infer_type("machine_id", pd.Series(["MACH-01", "MACH-01"])) == "IDENTITY"
assert infer_type("flag", pd.Series([0, 1, 1])) == "BOOLEAN"
assert infer_type("n", pd.Series([3, 5])) == "INTEGER"
assert infer_type("t", pd.Series([1.5, 2.0])) == "FLOAT"
assert infer_type("d", pd.Series(["2025-06-01", "2025-06-02"])) == "DATE"
assert infer_type("ts", pd.Series(["2025-06-01 00:00:00"])) == "DATETIME"
assert infer_type("h", pd.Series(["05:42", "21:08"])) == "TIME"
assert infer_type("c", pd.Series(["abc", "2025-06-01"])) == "TEXT"
mini_sql = """
CREATE TABLE t (code VARCHAR(8) PRIMARY KEY, d DATE, v NUMERIC(4,2), n INTEGER, ok BOOLEAN DEFAULT TRUE);
INSERT INTO t (code, d, v, n) VALUES ('A', '2025-01-01', 1.5, 3), ('B', NULL, 2.5, 4);
"""
df_t, types_t = parse_sql_tables(mini_sql)["t"]
assert types_t == {"code": "IDENTITY", "d": "DATE", "v": "FLOAT", "n": "INTEGER", "ok": "BOOLEAN"}
assert len(df_t) == 2 and list(df_t.columns) == ["code", "d", "v", "n"]
assert df_t["v"].mean() == 2.0 and df_t["d"].isna().sum() == 1
print("OK")

## 7. Génération des cartes

**Entrée** : tous les `*.csv` puis tous les `*.sql*` de `datas/`. Aucun nom de fichier n'est écrit dans le code :
un nouveau fichier déposé dans `datas/` aura sa carte automatiquement.

**Sortie** : le chemin de chaque carte écrite (une par fichier).

In [ ]:
for data_file in sorted(DATA_DIR.glob("*.csv")) + sorted(DATA_DIR.glob("*.sql*")):
    print(write_card(data_file))